# Cinema データセットの探索と興行収入予測モデルの訓練

このノートブックでは、Cinema データセットを使った回帰問題を TDD で実装します。

## 1. 環境設定とパスの確認

In [1]:
import java.io.File

// 現在のワーキングディレクトリを確認
val currentDir = File(".").absolutePath
println("現在のディレクトリ: $currentDir")

// データファイルのパスを自動検出
val possiblePaths = listOf(
    "src/main/resources/data/cinema.csv",           // app/kotlin から実行
    "../src/main/resources/data/cinema.csv",        // notebook から実行
    "app/kotlin/src/main/resources/data/cinema.csv", // プロジェクトルートから実行
    "../../src/main/resources/data/cinema.csv"      // さらに深い場所から実行
)

val dataPath = possiblePaths.firstOrNull { File(it).exists() }
    ?: error("cinema.csv が見つかりません。現在のディレクトリ: $currentDir")

println("データファイル: $dataPath")
println("ファイル存在確認: ${File(dataPath).exists()}")

現在のディレクトリ: C:\Users\PC202411-1\IdeaProjects\case-study-game-dev\app\kotlin\notebook\.
データファイル: ../src/main/resources/data/cinema.csv
ファイル存在確認: true


## 2. 依存関係の設定

In [2]:
// Smile ML ライブラリと BLAS 実装の依存関係を追加
@file:Repository("https://jitpack.io")
@file:DependsOn("com.github.haifengl:smile-core:3.0.2")
@file:DependsOn("com.github.haifengl:smile-kotlin:3.0.2")
@file:DependsOn("org.bytedeco:openblas-platform:0.3.21-1.5.8")

println("依存関係の設定完了")

依存関係の設定完了


## 3. CinemaPredictor クラスの定義とデータ読み込み

In [3]:
import smile.data.DataFrame as SmileDataFrame
import smile.data.formula.Formula
import smile.data.vector.DoubleVector
import smile.regression.LinearModel
import smile.regression.OLS
import java.io.Serializable
import kotlin.math.abs
import kotlin.math.pow
import kotlin.math.sqrt

/**
 * 映画興行収入を予測する線形回帰モデル
 */
class CinemaPredictor : Serializable {
    
    var model: LinearModel? = null
        private set
    
    companion object {
        private const val serialVersionUID = 1L
        private const val OUTLIER_SNS2_THRESHOLD = 1000.0
        private const val OUTLIER_SALES_THRESHOLD = 8500.0
        private const val FEATURE_SNS1_IDX = 0
        private const val FEATURE_SNS2_IDX = 1
        private const val FEATURE_ACTOR_IDX = 2
        private const val FEATURE_ORIGINAL_IDX = 3
    }
    
    /**
     * CSV ファイルからデータを読み込む
     */
    fun loadData(filePath: String, removeOutliers: Boolean = true): Pair<Array<DoubleArray>, DoubleArray> {
        val file = java.io.File(filePath)
        require(file.exists()) { "File not found: $filePath" }
        
        val lines = file.readLines()
        require(lines.isNotEmpty()) { "Empty file: $filePath" }
        
        val headerLine = lines[0].replace("\uFEFF", "").trim()
        val header = headerLine.split(",").map { it.trim() }
        
        val sns1Idx = header.indexOfFirst { it.equals("SNS1", ignoreCase = true) }
        val sns2Idx = header.indexOfFirst { it.equals("SNS2", ignoreCase = true) }
        val actorIdx = header.indexOfFirst { it.equals("actor", ignoreCase = true) }
        val originalIdx = header.indexOfFirst { it.equals("original", ignoreCase = true) }
        val salesIdx = header.indexOfFirst { it.equals("sales", ignoreCase = true) }
        
        require(sns1Idx >= 0 && sns2Idx >= 0 && actorIdx >= 0 && originalIdx >= 0 && salesIdx >= 0) {
            "Required columns not found in CSV"
        }
        
        // 第1パス: 平均値を計算
        val sns1Values = mutableListOf<Double>()
        val sns2Values = mutableListOf<Double>()
        val actorValues = mutableListOf<Double>()
        val originalValues = mutableListOf<Double>()
        
        for (i in 1 until lines.size) {
            val values = lines[i].split(",")
            if (values.size != header.size) continue
            
            values.getOrNull(sns1Idx)?.trim()?.toDoubleOrNull()?.let { sns1Values.add(it) }
            values.getOrNull(sns2Idx)?.trim()?.toDoubleOrNull()?.let { sns2Values.add(it) }
            values.getOrNull(actorIdx)?.trim()?.toDoubleOrNull()?.let { actorValues.add(it) }
            values.getOrNull(originalIdx)?.trim()?.toDoubleOrNull()?.let { originalValues.add(it) }
        }
        
        val sns1Mean = if (sns1Values.isNotEmpty()) sns1Values.average() else 0.0
        val sns2Mean = if (sns2Values.isNotEmpty()) sns2Values.average() else 0.0
        val actorMean = if (actorValues.isNotEmpty()) actorValues.average() else 0.0
        val originalMean = if (originalValues.isNotEmpty()) originalValues.average() else 0.0
        
        // 第2パス: データ行を読み込み
        val dataRows = mutableListOf<Pair<DoubleArray, Double>>()
        
        for (i in 1 until lines.size) {
            val values = lines[i].split(",")
            if (values.size != header.size) continue
            
            try {
                val sns1 = values[sns1Idx].trim().toDoubleOrNull() ?: sns1Mean
                val sns2 = values[sns2Idx].trim().toDoubleOrNull() ?: sns2Mean
                val actor = values[actorIdx].trim().toDoubleOrNull() ?: actorMean
                val original = values[originalIdx].trim().toDoubleOrNull() ?: originalMean
                val sales = values[salesIdx].trim().toDoubleOrNull() ?: continue
                
                if (removeOutliers && sns2 > OUTLIER_SNS2_THRESHOLD && sales < OUTLIER_SALES_THRESHOLD) {
                    continue
                }
                
                val features = doubleArrayOf(sns1, sns2, actor, original)
                dataRows.add(Pair(features, sales))
            } catch (e: Exception) {
                continue
            }
        }
        
        val X = Array(dataRows.size) { dataRows[it].first }
        val y = DoubleArray(dataRows.size) { dataRows[it].second }
        
        return Pair(X, y)
    }
    
    /**
     * モデルを訓練する
     */
    fun train(X: Array<DoubleArray>, y: DoubleArray) {
        require(X.isNotEmpty() && y.isNotEmpty()) { "Training data cannot be empty" }
        require(X.size == y.size) { "X and y must have the same length" }
        
        val data = SmileDataFrame.of(
            DoubleVector.of("SNS1", X.map { it[FEATURE_SNS1_IDX] }.toDoubleArray()),
            DoubleVector.of("SNS2", X.map { it[FEATURE_SNS2_IDX] }.toDoubleArray()),
            DoubleVector.of("actor", X.map { it[FEATURE_ACTOR_IDX] }.toDoubleArray()),
            DoubleVector.of("original", X.map { it[FEATURE_ORIGINAL_IDX] }.toDoubleArray()),
            DoubleVector.of("sales", y)
        )
        
        val formula = Formula.lhs("sales")
        model = OLS.fit(formula, data)
    }
    
    /**
     * 予測を実行する
     */
    fun predict(X: Array<DoubleArray>): DoubleArray {
        requireNotNull(model) { "Model has not been trained yet" }
        return X.map { x -> model!!.predict(x) }.toDoubleArray()
    }
    
    /**
     * モデルの性能を評価する
     */
    fun evaluate(X: Array<DoubleArray>, y: DoubleArray): Map<String, Double> {
        requireNotNull(model) { "Model has not been trained yet" }
        
        val predictions = predict(X)
        
        val yMean = y.average()
        val ssTot = y.sumOf { (it - yMean).pow(2) }
        val ssRes = y.zip(predictions).sumOf { (actual, pred) -> (actual - pred).pow(2) }
        val r2Score = 1.0 - (ssRes / ssTot)
        
        val mae = y.zip(predictions).sumOf { (actual, pred) -> abs(actual - pred) } / y.size
        val mse = y.zip(predictions).sumOf { (actual, pred) -> (actual - pred).pow(2) } / y.size
        val rmse = sqrt(mse)
        
        return mapOf(
            "r2Score" to r2Score,
            "mae" to mae,
            "rmse" to rmse
        )
    }
    
    fun saveModel(filePath: String) {
        requireNotNull(model) { "No trained model to save" }
        java.io.ObjectOutputStream(java.io.FileOutputStream(filePath)).use { oos ->
            oos.writeObject(model)
        }
    }
}

// モデルの作成
val predictor = CinemaPredictor()

// データの読み込み（外れ値除去あり）
val (X, y) = predictor.loadData(dataPath, removeOutliers = true)

println("=".repeat(60))
println("データの概要")
println("=".repeat(60))
println("サンプル数: ${X.size}")
println("特徴量数: ${X[0].size}")
println("特徴量: SNS1, SNS2, actor, original")
println()

// データの最初の5行を表示
println("最初の5サンプル:")
println("SNS1, SNS2, actor, original, sales")
for (i in 0 until minOf(5, X.size)) {
    println("%.1f, %.1f, %.1f, %.0f, %.1f".format(X[i][0], X[i][1], X[i][2], X[i][3], y[i]))
}

データの概要
サンプル数: 99
特徴量数: 4
特徴量: SNS1, SNS2, actor, original

最初の5サンプル:
SNS1, SNS2, actor, original, sales
291.0, 1044.0, 8809.0, 0, 9731.0
363.0, 568.0, 10290.7, 1, 10210.0
158.0, 431.0, 6340.4, 1, 8227.0
261.0, 578.0, 8250.5, 0, 9658.0
209.0, 683.0, 10908.5, 0, 9286.0


## 4. Lets-Plot の初期化

In [4]:
%use lets-plot

## 5. データの可視化

### 5.1 興行収入の分布

In [5]:
// 興行収入の分布をヒストグラムで可視化
val salesValues = y.toList()

val p = letsPlot() + 
    geomHistogram(bins = 20) { x = salesValues }

p

8,000 
 
 
 
 
 
 
 
 
 8,500 
 
 
 
 
 
 
 
 
 9,000 
 
 
 
 
 
 
 
 
 9,500 
 
 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 
 
 10,500 
 
 
 
 
 
 
 
 
 11,000 
 
 
 
 
 
 
 
 
 11,500 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 x

### 5.2 SNS1 と興行収入の関係

In [6]:
// SNS1 vs 興行収入の散布図
val sns1Data = X.map { it[0] }
val salesData = y.toList()

val p = letsPlot() + 
    geomPoint(size = 3.0, alpha = 0.7) { 
        x = sns1Data
        y = salesData
    }

p

0 
 
 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 400 
 
 
 
 
 
 
 
 
 600 
 
 
 
 
 
 
 
 
 800 
 
 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 
 
 
 
 8,000 
 
 
 
 
 
 
 8,500 
 
 
 
 
 
 
 9,000 
 
 
 
 
 
 
 9,500 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 10,500 
 
 
 
 
 
 
 11,000 
 
 
 
 
 
 
 11,500 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x

### 5.3 SNS2 と興行収入の関係

In [7]:
// SNS2 vs 興行収入の散布図
val sns2Data = X.map { it[1] }

val p = letsPlot() + 
    geomPoint(size = 3.0, alpha = 0.7) { 
        x = sns2Data
        y = salesData
    }

p

0 
 
 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 400 
 
 
 
 
 
 
 
 
 600 
 
 
 
 
 
 
 
 
 800 
 
 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 
 
 1,200 
 
 
 
 
 
 
 
 
 1,400 
 
 
 
 
 
 
 
 
 
 
 8,000 
 
 
 
 
 
 
 8,500 
 
 
 
 
 
 
 9,000 
 
 
 
 
 
 
 9,500 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 10,500 
 
 
 
 
 
 
 11,000 
 
 
 
 
 
 
 11,500 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x

### 5.4 actor と興行収入の関係

In [8]:
// actor vs 興行収入の散布図
val actorData = X.map { it[2] }

val p = letsPlot() + 
    geomPoint(size = 3.0, alpha = 0.7) { 
        x = actorData
        y = salesData
    }

p

6,000 
 
 
 
 
 
 
 
 
 7,000 
 
 
 
 
 
 
 
 
 8,000 
 
 
 
 
 
 
 
 
 9,000 
 
 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 
 
 11,000 
 
 
 
 
 
 
 
 
 12,000 
 
 
 
 
 
 
 
 
 13,000 
 
 
 
 
 
 
 
 
 
 
 8,000 
 
 
 
 
 
 
 8,500 
 
 
 
 
 
 
 9,000 
 
 
 
 
 
 
 9,500 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 10,500 
 
 
 
 
 
 
 11,000 
 
 
 
 
 
 
 11,500 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x

## 6. 基本統計量の確認

In [9]:
val featureNames = listOf("SNS1", "SNS2", "actor", "original")

println("特徴量の基本統計量:")
println("-".repeat(60))

featureNames.forEachIndexed { idx, name ->
    val values = X.map { it[idx] }
    val min = values.minOrNull() ?: 0.0
    val max = values.maxOrNull() ?: 0.0
    val mean = values.average()
    val sorted = values.sorted()
    val median = if (sorted.size % 2 == 0) {
        (sorted[sorted.size / 2 - 1] + sorted[sorted.size / 2]) / 2.0
    } else {
        sorted[sorted.size / 2]
    }
    
    println("%-10s: 最小=%.1f, 最大=%.1f, 平均=%.1f, 中央値=%.1f".format(name, min, max, mean, median))
}

println()
println("興行収入の基本統計量:")
println("-".repeat(60))
val salesMin = y.minOrNull() ?: 0.0
val salesMax = y.maxOrNull() ?: 0.0
val salesMean = y.average()
val salesSorted = y.sorted()
val salesMedian = if (salesSorted.size % 2 == 0) {
    (salesSorted[salesSorted.size / 2 - 1] + salesSorted[salesSorted.size / 2]) / 2.0
} else {
    salesSorted[salesSorted.size / 2]
}
val salesStd = sqrt(y.map { (it - salesMean) * (it - salesMean) }.average())

println("sales:      最小=%.1f, 最大=%.1f, 平均=%.1f, 中央値=%.1f, 標準偏差=%.1f".format(
    salesMin, salesMax, salesMean, salesMedian, salesStd
))

特徴量の基本統計量:
------------------------------------------------------------
SNS1      : 最小=0.0, 最大=1000.0, 平均=373.0, 中央値=353.0
SNS2      : 最小=0.0, 最大=1500.0, 平均=653.0, 中央値=653.0
actor     : 最小=5702.8, 最大=12665.1, 平均=9834.1, 中央値=9991.7
original  : 最小=0.0, 最大=1.0, 平均=0.5, 中央値=1.0

興行収入の基本統計量:
------------------------------------------------------------
sales:      最小=7869.0, 最大=11405.0, 平均=9901.4, 中央値=9966.0, 標準偏差=776.3


## 7. モデルの訓練

In [10]:
println("モデルの訓練中...")
val startTime = System.currentTimeMillis()
predictor.train(X, y)
val trainingTime = System.currentTimeMillis() - startTime

println("訓練完了！")
println("訓練時間: ${trainingTime}ms")
println()

モデルの訓練中...
訓練完了！
訓練時間: 422ms



## 8. モデルの評価

In [11]:
val metrics = predictor.evaluate(X, y)

println("=".repeat(60))
println("モデルの評価結果")
println("=".repeat(60))
println("R² スコア:        %.4f".format(metrics["r2Score"]))
println("MAE（平均絶対誤差）: %.2f 万円".format(metrics["mae"]))
println("RMSE（二乗平均平方根誤差）: %.2f 万円".format(metrics["rmse"]))
println()

モデルの評価結果
R² スコア:        0.7721
MAE（平均絶対誤差）: 296.47 万円
RMSE（二乗平均平方根誤差）: 370.61 万円



## 9. 予測 vs 実測の可視化

In [12]:
// 予測値を計算
val predictions = predictor.predict(X)

// 予測 vs 実測の散布図
val actualData = y.toList()
val predData = predictions.toList()

val p = letsPlot() + 
    geomPoint(size = 3.0, alpha = 0.7) { 
        x = actualData
        y = predData
    } +
    geomABLine(slope = 1.0, intercept = 0.0, color = "red", linetype = "dashed")

p

8,000 
 
 
 
 
 
 
 
 
 8,500 
 
 
 
 
 
 
 
 
 9,000 
 
 
 
 
 
 
 
 
 9,500 
 
 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 
 
 10,500 
 
 
 
 
 
 
 
 
 11,000 
 
 
 
 
 
 
 
 
 11,500 
 
 
 
 
 
 
 
 
 
 
 8,500 
 
 
 
 
 
 
 9,000 
 
 
 
 
 
 
 9,500 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 10,500 
 
 
 
 
 
 
 11,000 
 
 
 
 
 
 
 11,500 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x

## 10. 予測と実測の比較（最初の10件）

In [13]:
println("予測と実測の比較（最初の10件）:")
println("実測値    予測値    誤差     誤差率")
println("-".repeat(50))

(0 until minOf(10, X.size)).forEach { i ->
    val actual = y[i]
    val pred = predictions[i]
    val error = actual - pred
    val errorRate = abs(error) / actual * 100
    println("%7.1f   %7.1f   %+7.1f   %5.1f%%".format(actual, pred, error, errorRate))
}

予測と実測の比較（最初の10件）:
実測値    予測値    誤差     誤差率
--------------------------------------------------
 9731.0    9591.9    +139.1     1.4%
10210.0   10094.5    +115.5     1.1%
 8227.0    8691.1    -464.1     5.6%
 9658.0    9169.6    +488.4     5.1%
 9286.0    9894.4    -608.4     6.6%
 9574.0    9768.8    -194.8     2.0%
 7869.0    8898.7   -1029.7    13.1%
 9804.0   10238.3    -434.3     4.4%
 9023.0    9175.9    -152.9     1.7%
 9229.0    9393.7    -164.7     1.8%


## 11. 個別予測の例

In [14]:
println("個別予測の例:")
println("-".repeat(60))

val testSamples = listOf(
    Pair(doubleArrayOf(500.0, 800.0, 20.0, 1.0), "低〜中程度のSNS露出、オリジナル作品"),
    Pair(doubleArrayOf(1200.0, 1500.0, 50.0, 0.0), "高いSNS露出、人気俳優出演"),
    Pair(doubleArrayOf(300.0, 400.0, 10.0, 0.0), "低いSNS露出、新人俳優")
)

testSamples.forEachIndexed { i, (sample, description) ->
    val prediction = predictor.predict(arrayOf(sample))[0]
    println("サンプル ${i+1} ($description):")
    println("  SNS1=%.0f, SNS2=%.0f, actor=%.0f, original=%.0f".format(
        sample[0], sample[1], sample[2], sample[3]
    ))
    println("  予測興行収入: %.1f 万円".format(prediction))
    println()
}

個別予測の例:
------------------------------------------------------------
サンプル 1 (低〜中程度のSNS露出、オリジナル作品):
  SNS1=500, SNS2=800, actor=20, original=1
  予測興行収入: 7536.0 万円

サンプル 2 (高いSNS露出、人気俳優出演):
  SNS1=1200, SNS2=1500, actor=50, original=0
  予測興行収入: 8481.0 万円

サンプル 3 (低いSNS露出、新人俳優):
  SNS1=300, SNS2=400, actor=10, original=0
  予測興行収入: 6850.9 万円



## 12. モデルの保存

In [15]:
// モデル保存先のパスを構築
val modelDir = when {
    File("model").exists() || File(".").resolve("model").parentFile.exists() -> "model"
    File("../model").parentFile.exists() -> "../model"
    File("app/kotlin/model").parentFile.exists() -> "app/kotlin/model"
    else -> "model"
}

val modelPath = "$modelDir/cinema_model.ser"
File(modelPath).parentFile?.mkdirs()

predictor.saveModel(modelPath)

println("=".repeat(60))
println("モデルの保存完了")
println("=".repeat(60))
println("保存先: $modelPath")
println("ファイルサイズ: ${File(modelPath).length()} bytes")

モデルの保存完了
保存先: model/cinema_model.ser
ファイルサイズ: 3706 bytes


## まとめ

このノートブックでは、Cinema データセットを使った回帰問題を TDD で実装しました。

### 実施内容

1. **データの読み込みと探索**: CSV からのデータ読み込み、欠損値の平均値補完、外れ値除去
2. **データの可視化**: ヒストグラム、散布図による分布確認と相関分析
3. **モデルの訓練**: 線形回帰モデル（OLS）の訓練
4. **モデルの評価**: R²、MAE、RMSE の計算
5. **予測の実行**: 新しいデータでの興行収入予測
6. **モデルの保存**: モデルの永続化

### 次のステップ

- データを訓練用とテスト用に分割して汎化性能をチェック
- クロスバリデーションで性能を評価
- 他のデータセット（Survived、Boston）に挑戦
- Web API 化（Ktor による REST API 実装）

---

お疲れ様でした！線形回帰による機械学習開発の基礎を習得しました！